In [1]:
import os
import requests
import pandas as pd
import duckdb

In [2]:
url = "https://www.data.gouv.fr/api/1/datasets/r/2ce43ade-8d2c-4d1d-81da-ca06c82abc68"
response = requests.get(url)
with open("../data/raw/pharmacies.csv", "wb") as f:
    f.write(response.content)


In [3]:

df = pd.read_csv("../data/raw/pharmacies.csv", sep=";", dtype=str,skiprows=1,header = None)

In [4]:
df

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,structureet,010000024,010780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,NaN,NaN,900,RTE,DE PARIS,...,26010004500012,8610Z,03,ARS établissements Publics de santé dotation g...,1,Etablissement public de santé,1979-02-13,1979-02-13,2020-02-04,NaN
1,structureet,010000032,010780062,CH BUGEY SUD,CENTRE HOSPITALIER BUGEY SUD,NaN,NaN,700,AV,DE NARVIK,...,26010003700068,8610Z,03,ARS établissements Publics de santé dotation g...,1,Etablissement public de santé,1901-01-01,1901-01-01,2021-07-07,NaN
2,structureet,010000065,010780096,CH DE TREVOUX - MONTPENSIER,CENTRE HOSPITALIER DE TREVOUX - MONTPENSIER,NaN,NaN,14,R,DE L'HOPITAL,...,26010028400017,8610Z,03,ARS établissements Publics de santé dotation g...,1,Etablissement public de santé,1901-01-01,1901-01-01,2018-01-12,NaN
3,structureet,010000081,010780112,CH DU PAYS DE GEX,CENTRE HOSPITALIER DU PAYS DE GEX,NaN,NaN,160,R,MARC PANISSOD,...,26010010200011,8610Z,03,ARS établissements Publics de santé dotation g...,1,Etablissement public de santé,1901-01-01,1901-01-01,2020-02-04,NaN
4,structureet,010000099,010780120,CH DE MEXIMIEUX,CENTRE HOSPITALIER DE MEXIMIEUX,NaN,NaN,13,AV,DU DOCTEUR BOYER,...,26010013600019,8610Z,03,ARS établissements Publics de santé dotation g...,1,Etablissement public de santé,1945-01-01,1945-01-01,2020-06-30,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102538,structureet,980503320,980503312,OUNONO NA MAECHA,NaN,NaN,NaN,729,R,DE L'AVENIR,...,52858351100027,NaN,08,Président du Conseil Départemental,NaN,NaN,2023-07-25,2023-06-22,2025-09-26,NaN
102539,structureet,980503346,980503338,USAIDIYA,NaN,NaN,NaN,28,R,BABOU SALAMA,...,91320786600017,NaN,08,Président du Conseil Départemental,NaN,NaN,2023-09-04,2022-11-24,2025-09-26,NaN
102540,structureet,980503395,980503387,LA MAISON DU BONHEUR,NaN,NaN,NaN,2,R,BACARI DJOUMOI,...,94210204700019,NaN,08,Président du Conseil Départemental,NaN,NaN,2025-03-20,2025-01-17,2025-10-30,NaN
102541,structureet,980600027,980600019,HOPITAL DE SIA,NaN,NaN,NaN,NaN,NaN,NaN,...,13000323900014,NaN,99,Indéterminé,1,Etablissement public de santé,2024-10-30,2024-10-30,2024-10-30,NaN


In [4]:
df.iloc[:,19].head(50)

0                             Centre Hospitalier (C.H.)
1                             Centre Hospitalier (C.H.)
2                             Centre Hospitalier (C.H.)
3                  Centre hospitalier, ex Hôpital local
4                  Centre hospitalier, ex Hôpital local
5                  Centre hospitalier, ex Hôpital local
6                             Centre Hospitalier (C.H.)
7                             Centre Hospitalier (C.H.)
8                             Centre Hospitalier (C.H.)
9                             Centre Hospitalier (C.H.)
10                            Centre Hospitalier (C.H.)
11    Centre Hospitalier Spécialisé lutte Maladies M...
12                                 Résidences autonomie
13    Etablissement et Service d'Aide par le Travail...
14                            Club Equipe de Prévention
15                                 Résidences autonomie
16    Service de Soins Infirmiers A Domicile (S.S.I....
17                     Laboratoire de Biologie M

In [5]:
df = df.iloc[:, [15, 19]]

In [6]:
df.rename(columns={19: "type", 15: "code_insee"}, inplace=True)

In [8]:
df = df.loc[df["type"].str.startswith("Phar")].reset_index(drop=True)

In [7]:
df

,code_insee,type
0,01440 VIRIAT,Centre Hospitalier (C.H.)
1,01300 BELLEY,Centre Hospitalier (C.H.)
2,01606 TREVOUX CEDEX,Centre Hospitalier (C.H.)
3,01174 GEX CEDEX,"Centre hospitalier, ex Hôpital local"
4,01800 MEXIMIEUX,"Centre hospitalier, ex Hôpital local"
...,...,...
102538,97650 BANDRABOUA,Service autonomie aide (SAA)
102539,97600 MAMOUDZOU,Service autonomie aide (SAA)
102540,97650 BANDRABOUA,Lieux de Vie et d'Accueil
102541,98600 UVEA,Centre Hospitalier (C.H.)


In [8]:
def code_postal(string):
    return string.split(" ")[0]

In [9]:
df['code_postal'] = df['code_insee'].apply(lambda x: x.split(" ")[0])

In [10]:
df.drop(columns=["code_insee"], inplace=True)

In [11]:
df.head(50)

,type,code_postal
0,Centre Hospitalier (C.H.),01440
1,Centre Hospitalier (C.H.),01300
2,Centre Hospitalier (C.H.),01606
3,"Centre hospitalier, ex Hôpital local",01174
4,"Centre hospitalier, ex Hôpital local",01800
5,"Centre hospitalier, ex Hôpital local",01190
6,Centre Hospitalier (C.H.),01290
7,Centre Hospitalier (C.H.),01140
8,Centre Hospitalier (C.H.),01110
9,Centre Hospitalier (C.H.),01110


In [26]:
df[(df['code_postal'].str.startswith("75")) & (df['type'].str.startswith("Phar"))]

,type,code_postal
72876,Pharmacie d'Officine,75000
72877,Pharmacie d'Officine,75000
72878,Pharmacie d'Officine,75000
72879,Pharmacie d'Officine,75000
72880,Pharmacie d'Officine,75000
...,...,...
73949,Pharmacie d'Officine,75000
73993,Pharmacie d'Officine,75000
74036,Pharmacie d'Officine,75000
74074,Pharmacie d'Officine,75000


In [25]:
#On change le code postal commençant par 75 par 75000
df['code_postal'] = df['code_postal'].apply(lambda x: '75000' if x.startswith('75') else x)

In [20]:
df_epci = pd.read_csv("../data/processed/epci_membres.csv", sep=",", dtype=str)

In [21]:
df_epci

,code_insee,nom,pop_tot_commune,pop_mun_commune,siren,epci_nom,epci_type,epci_modeFinancement,total_pop_tot,total_pop_mun,superficie_hectare,superficie_km2,dept,bassin_vie
0,01304,Pont-d'Ain,2912,2862,200029999,CC Rives de l'Ain - Pays du Cerdon,CC,FPU,15156,14873,1122.0,11.0,01,01304
1,01199,Jujurieux,2246,2209,200029999,CC Rives de l'Ain - Pays du Cerdon,CC,FPU,15156,14873,1548.0,15.0,01,01004
2,01363,Saint-Jean-le-Vieux,1873,1799,200029999,CC Rives de l'Ain - Pays du Cerdon,CC,FPU,15156,14873,1517.0,15.0,01,01004
3,01314,Priay,1826,1803,200029999,CC Rives de l'Ain - Pays du Cerdon,CC,FPU,15156,14873,1571.0,16.0,01,01004
4,01273,Neuville-sur-Ain,1823,1798,200029999,CC Rives de l'Ain - Pays du Cerdon,CC,FPU,15156,14873,1991.0,20.0,01,01004
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34996,97407,Le Port,33940,33670,249740101,CA Territoire de la Côte Ouest (TCO),CA,FPU,221532,218990,1608.0,16.0,97,97415
34997,97423,Les Trois-Bassins,7207,7113,249740101,CA Territoire de la Côte Ouest (TCO),CA,FPU,221532,218990,4247.0,42.0,97,97415
34998,97411,Saint-Denis,157674,156149,249740119,CA Intercommunale du Nord de la Réunion (CINOR),CA,FPU,218801,216588,14150.0,142.0,97,97411
34999,97418,Sainte-Marie,36014,35584,249740119,CA Intercommunale du Nord de la Réunion (CINOR),CA,FPU,218801,216588,8864.0,89.0,97,97411


In [15]:
df_com = pd.read_csv("https://www.data.gouv.fr/api/1/datasets/r/f5df602b-3800-44d7-b2df-fa40a0350325")

/tmp/ipykernel_9503/1384171285.py:1: DtypeWarning: Columns (1,12,14,16,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  df_com = pd.read_csv("https://www.data.gouv.fr/api/1/datasets/r/f5df602b-3800-44d7-b2df-fa40a0350325")


In [21]:
df_com

,Unnamed: 0,code_insee,nom_standard,nom_sans_pronom,nom_a,nom_de,nom_sans_accent,nom_standard_majuscule,typecom,typecom_texte,...,longitude_mairie,latitude_centre,longitude_centre,grille_densite,grille_densite_texte,niveau_equipements_services,niveau_equipements_services_texte,gentile,url_wikipedia,url_villedereve
0,0,01001,L'Abergement-Clémenciat,Abergement-Clémenciat,à Abergement-Clémenciat,de l'Abergement-Clémenciat,l-abergement-clemenciat,L'ABERGEMENT-CLÉMENCIAT,COM,commune,...,4.921,46.153,4.926,6,Rural à habitat dispersé,0.0,communes non pôle,NaN,https://fr.wikipedia.org/wiki/fr:L'Abergement-...,https://villedereve.fr/ville/01001-l-abergemen...
1,1,01002,L'Abergement-de-Varey,Abergement-de-Varey,à Abergement-de-Varey,de l'Abergement-de-Varey,l-abergement-de-varey,L'ABERGEMENT-DE-VAREY,COM,commune,...,5.423,46.009,5.428,6,Rural à habitat dispersé,0.0,communes non pôle,"Abergementais, Abergementaises",https://fr.wikipedia.org/wiki/fr:L'Abergement-...,https://villedereve.fr/ville/01002-l-abergemen...
2,2,01004,Ambérieu-en-Bugey,Ambérieu-en-Bugey,à Ambérieu-en-Bugey,d'Ambérieu-en-Bugey,amberieu-en-bugey,AMBÉRIEU-EN-BUGEY,COM,commune,...,5.360,45.961,5.373,2,Centres urbains intermédiaires,3.0,centres structurants d'équipements et de services,"Ambarrois, Ambarroises",https://fr.wikipedia.org/wiki/fr:Ambérieu-en-B...,https://villedereve.fr/ville/01004-amberieu-en...
3,3,01005,Ambérieux-en-Dombes,Ambérieux-en-Dombes,à Ambérieux-en-Dombes,d'Ambérieux-en-Dombes,amberieux-en-dombes,AMBÉRIEUX-EN-DOMBES,COM,commune,...,4.903,45.996,4.912,5,Bourgs ruraux,1.0,centres locaux d'équipements et de services,Ambarrois,https://fr.wikipedia.org/wiki/fr:Ambérieux-en-...,https://villedereve.fr/ville/01005-amberieux-e...
4,4,01006,Ambléon,Ambléon,à Ambléon,d'Ambléon,ambleon,AMBLÉON,COM,commune,...,5.601,45.750,5.594,6,Rural à habitat dispersé,0.0,communes non pôle,Ambléonais,https://fr.wikipedia.org/wiki/fr:Ambléon,https://villedereve.fr/ville/01006-ambleon
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34930,34930,97613,M'Tsangamouji,M'Tsangamouji,à M'Tsangamouji,de M'Tsangamouji,m-tsangamouji,M'TSANGAMOUJI,COM,commune,...,45.084,-12.751,45.087,3,Petites villes,NaN,NaN,NaN,https://fr.wikipedia.org/wiki/fr:M'Tsangamouji,https://villedereve.fr/ville/97613-m-tsangamouji
34931,34931,97614,Ouangani,Ouangani,à Ouangani,d'Ouangani,ouangani,OUANGANI,COM,commune,...,45.139,-12.837,45.138,3,Petites villes,NaN,NaN,NaN,https://fr.wikipedia.org/wiki/fr:Ouangani,https://villedereve.fr/ville/97614-ouangani
34932,34932,97615,Pamandzi,Pamandzi,à Pamandzi,de Pamandzi,pamandzi,PAMANDZI,COM,commune,...,45.275,-12.796,45.284,2,Centres urbains intermédiaires,NaN,NaN,Pamandziens,https://fr.wikipedia.org/wiki/fr:Pamandzi,https://villedereve.fr/ville/97615-pamandzi
34933,34933,97616,Sada,Sada,à Sada,de Sada,sada,SADA,COM,commune,...,45.106,-12.861,45.119,2,Centres urbains intermédiaires,NaN,NaN,Sadois,https://fr.wikipedia.org/wiki/fr:Sada (Mayotte),https://villedereve.fr/ville/97616-sada


In [22]:
df_com['code_postal'] = df_com['code_postal'].astype(str).str.replace(".0", "", regex=False).str.zfill(5)

In [23]:
df_com[df_com['code_postal'].str.startswith("75")]

,Unnamed: 0,code_insee,nom_standard,nom_sans_pronom,nom_a,nom_de,nom_sans_accent,nom_standard_majuscule,typecom,typecom_texte,...,longitude_mairie,latitude_centre,longitude_centre,grille_densite,grille_densite_texte,niveau_equipements_services,niveau_equipements_services_texte,gentile,url_wikipedia,url_villedereve
29244,29244,75056,Paris,Paris,à Paris,de Paris,paris,PARIS,COM,commune,...,2.352,NaN,NaN,1,Grands centres urbains,4.0,centres majeurs d'équipements et de services,Parisien,https://fr.wikipedia.org/wiki/fr:Paris,https://villedereve.fr/ville/75056-paris


In [37]:
df_com[df_com['nom_standard'].str.startswith('Paris')]

,Unnamed: 0,code_insee,nom_standard,nom_sans_pronom,nom_a,nom_de,nom_sans_accent,nom_standard_majuscule,typecom,typecom_texte,...,longitude_mairie,latitude_centre,longitude_centre,grille_densite,grille_densite_texte,niveau_equipements_services,niveau_equipements_services_texte,gentile,url_wikipedia,url_villedereve
28102,28102,71343,Paris-l'Hôpital,Paris-l'Hôpital,à Paris-l'Hôpital,de Paris-l'Hôpital,paris-l-hopital,PARIS-L'HÔPITAL,COM,commune,...,4.639,46.913,4.640,6,Rural à habitat dispersé,0.0,communes non pôle,NaN,https://fr.wikipedia.org/wiki/fr:Paris-l'Hôpital,https://villedereve.fr/ville/71343-paris-l-hop...
29244,29244,75056,Paris,Paris,à Paris,de Paris,paris,PARIS,COM,commune,...,2.352,NaN,NaN,1,Grands centres urbains,4.0,centres majeurs d'équipements et de services,Parisien,https://fr.wikipedia.org/wiki/fr:Paris,https://villedereve.fr/ville/75056-paris
31941,31941,81202,Parisot,Parisot,à Parisot,de Parisot,parisot,PARISOT,COM,commune,...,1.831,43.805,1.836,6,Rural à habitat dispersé,1.0,centres locaux d'équipements et de services,Parisotains,https://fr.wikipedia.org/wiki/fr:Parisot (Tarn),https://villedereve.fr/ville/81202-parisot
32197,32197,82137,Parisot,Parisot,à Parisot,de Parisot,parisot,PARISOT,COM,commune,...,1.857,44.262,1.866,7,Rural à habitat très dispersé,1.0,centres locaux d'équipements et de services,Parisotins,https://fr.wikipedia.org/wiki/fr:Parisot (Tarn...,https://villedereve.fr/ville/82137-parisot


In [26]:
query = """
SELECT
    df_com.epci_code AS id_epci,
    df_com.code_insee,
    'i066' AS id_indicator,
    df.code_postal,
    '2025' AS annee
FROM df
LEFT JOIN df_com
    ON df.code_postal = df_com.code_postal
"""

result = duckdb.sql(query)
result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬────────────┬──────────────┬─────────────┬─────────┐
│  id_epci  │ code_insee │ id_indicator │ code_postal │  annee  │
│  varchar  │  varchar   │   varchar    │   varchar   │ varchar │
├───────────┼────────────┼──────────────┼─────────────┼─────────┤
│ 240100883 │ 01379      │ i066         │ 01500       │ 2025    │
│ 240100883 │ 01379      │ i066         │ 01500       │ 2025    │
│ 240100883 │ 01379      │ i066         │ 01500       │ 2025    │
│ 240100883 │ 01379      │ i066         │ 01500       │ 2025    │
│ 200069193 │ 01443      │ i066         │ 01330       │ 2025    │
│ 240100883 │ 01379      │ i066         │ 01500       │ 2025    │
│ 200042935 │ 01283      │ i066         │ 01100       │ 2025    │
│ 200042497 │ 01446      │ i066         │ 01480       │ 2025    │
│ 200040350 │ 01452      │ i066         │ 01510       │ 2025    │
│ 200071751 │ 01387      │ i066         │ 01340       │ 2025    │
│     ·     │   ·        │  ·           │   ·         │  ·      │
│     ·   

In [31]:
df_epci_commune = duckdb.sql(""" SELECT  siren, epci_type, TRY_CAST(REPLACE(total_pop_tot,' ','') AS INTEGER) as total_pop FROM df_epci WHERE epci_type = 'Commune'""")

In [32]:
df_epci_commune

┌─────────┬───────────┬───────────┐
│  siren  │ epci_type │ total_pop │
│ varchar │  varchar  │   int32   │
├─────────┼───────────┼───────────┤
│ 75056   │ Commune   │   2129257 │
└─────────┴───────────┴───────────┘

In [30]:
df_epci_pop_tot

┌───────────┬───────────┬───────────┐
│   siren   │ epci_type │ total_pop │
│  varchar  │  varchar  │   int32   │
├───────────┼───────────┼───────────┤
│ 200029999 │ CC        │     15156 │
│ 200029999 │ CC        │     15156 │
│ 200029999 │ CC        │     15156 │
│ 200029999 │ CC        │     15156 │
│ 200029999 │ CC        │     15156 │
│ 200029999 │ CC        │     15156 │
│ 200029999 │ CC        │     15156 │
│ 200029999 │ CC        │     15156 │
│ 200029999 │ CC        │     15156 │
│ 200029999 │ CC        │     15156 │
│     ·     │ ·         │       ·   │
│     ·     │ ·         │       ·   │
│     ·     │ ·         │       ·   │
│ 200040277 │ CA        │    118593 │
│ 200040277 │ CA        │    118593 │
│ 200040277 │ CA        │    118593 │
│ 200040277 │ CA        │    118593 │
│ 200040277 │ CA        │    118593 │
│ 200040277 │ CA        │    118593 │
│ 200040277 │ CA        │    118593 │
│ 200040277 │ CA        │    118593 │
│ 200040277 │ CA        │    118593 │
│ 200040277 

In [106]:
query_final = """
SELECT 
    result.id_epci,
    result.id_indicator,
    ROUND((result.valeur_brute/ df_epci_pop_tot.total_pop) * 10000, 2) AS valeur_brute,
    result.annee
FROM df_epci_pop_tot
LEFT JOIN result 
ON result.id_epci = df_epci_pop_tot.siren
WHERE result.id_epci IS NOT NULL
"""

df_densite_parma_i066 = duckdb.sql(query_final)


In [107]:
df_densite_parma_i066 = df_densite_parma_i066.df()

In [109]:
df_densite_parma_i066["valeur_brute"].min()

1.06